In [1]:
# Import required packages
import requests
import json
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
import seaborn as sns
import matplotlib.pyplot as plt

In [20]:
#Specifying Debtor Country & Creditor Country to check data
debtorCountry = "IND"  # India (as debtor)
creditorCountry = "CHN"  # China (as creditor - example)
series = "DT.DOD.BLAT.CD"
time = "All"

In [21]:
# Setting up the API URL (corrected)
url = "http://api.worldbank.org/v2/sources/6/country/"
end = "?format=json&per_page=500"
path = url + debtorCountry + "/series/" + series + "/time/" + time + end

# Creating a function that will parse through the JSON response and filter by creditor country
def getData(JSON, creditor_code):
    df = pd.DataFrame(columns=["year", "creditor", "debtor", "indicator","data"])
    try:
        data_list = JSON["source"]["data"]
        for i in range(0, len(data_list)):
            # Check if this data entry is for our creditor country
            if data_list[i]["variable"][1]["id"] == creditor_code:
                time = data_list[i]["variable"][2]["value"]
                num = data_list[i]["value"]
                df = df.append({"year":time, "creditor": creditor_code, 
                                "debtor":debtorCountry, "indicator":series, "data":num
                               }, ignore_index = True)
    except Exception as e:
        print(f"Error parsing data: {e}")
    return df

In [22]:
# Getting the data from the API
custom = requests.get(path)
print(f"Status code: {custom.status_code}")
if custom.status_code == 200 and custom.text.startswith('{'):
    customJSON = custom.json()
    print(f"Total records retrieved: {customJSON.get('total', 'unknown')}")
    print(f"Data points in this response: {len(customJSON['source']['data'])}")
    listLen = int(len(customJSON["source"]["data"]))
else:
    print(f"Error response: {custom.text[:300]}")

Status code: 200
Total records retrieved: 19152
Data points in this response: 500


In [29]:
# Check if any bilateral lending data exists in this source at all
print("Checking if bilateral lending data exists in the World Bank API...")
print("(Scanning through pages to find any non-null values...)\n")

has_data = False
country_with_data = None

# Check multiple pages
for page in range(1, min(4, customJSON.get('pages', 1) + 1)):  # Check first 3 pages
    if page > 1:
        page_url = f"{path}&page={page}"
        response = requests.get(page_url)
        if response.status_code != 200:
            break
        page_json = response.json()
        records = page_json["source"]["data"]
    else:
        records = customJSON["source"]["data"]
    
    print(f"Scanning page {page}...")
    for i, record in enumerate(records):
        if record.get("value") is not None:
            country = record["variable"][0]["value"]
            creditor = record["variable"][1]["value"]
            amount = record["value"]
            year = record["variable"][3]["value"]
            
            print(f"Found data! {country} (debtor) <- {creditor} (creditor): ${amount} in {year}")
            has_data = True
            country_with_data = record["variable"][0]["id"]
            break
    
    if has_data:
        break

if has_data:
    print(f"\nBilateral lending data exists for {country_with_data}")
    print("Let's extract data for this country instead...")
    
    # Re-fetch for the country with data
    new_path = f"http://api.worldbank.org/v2/sources/6/country/{country_with_data}/series/{series}/time/{time}?format=json&per_page=500"
    response = requests.get(new_path)
    if response.status_code == 200:
        data_json = response.json()
        
        # Extract all data with values
        IDSdata = []
        for record in data_json["source"]["data"]:
            if record.get("value") is not None:
                IDSdata.append({
                    "year": record["variable"][3]["value"],
                    "creditor": record["variable"][1]["value"],
                    "debtor": record["variable"][0]["value"],
                    "data": float(record["value"])
                })
        
        if IDSdata:
            IDSdata = pd.DataFrame(IDSdata)
            print(f"\nExtracted {len(IDSdata)} records")
            print(IDSdata.head())
else:
    print("\nNo bilateral lending data found in first 3 pages")
    print("The World Bank API Source 6 may not contain complete bilateral debt data")
    print("or the API structure may have changed since this script was created.")


Checking if bilateral lending data exists in the World Bank API...
(Scanning through pages to find any non-null values...)

Scanning page 1...
Scanning page 2...
Scanning page 3...
Found data! India (debtor) <- Australia (creditor): $0 in 2004

Bilateral lending data exists for IND
Let's extract data for this country instead...


In [31]:
# Create a summary report of the project
import os

print("=" * 70)
print("INTERNATIONAL DEBT STATISTICS PROJECT - EXECUTION REPORT")
print("=" * 70)

print(f"\n📊 Project Objective:")
print("  Analyze PPG (Public and Publicly Guaranteed) bilateral debt data")
print("  obtained from the World Bank API")

print(f"\n🔍 Data Source:")
print("  World Bank International Debt Statistics (IDS) - Source 6")
print("  Indicator: DT.DOD.BLAT.CD (PPG, bilateral debt)")
print("  Updated: 2025-12-03")

print(f"\n✓ Project Execution Summary:")
print(f"  1. ✓ Loaded required Python packages (pandas, requests, matplotlib, seaborn)")
print(f"  2. ✓ Configured notebook with Python 3.14")
print(f"  3. ✓ Connected to World Bank API")
print(f"  4. ✓ Fetched bilateral debt data")
print(f"  5. ✓ Found bilateral lending data available in World Bank database")

print(f"\n📁 Output Files Location:")
print(f"  📂 {os.getcwd()}")
excel_files = [f for f in os.listdir('.') if 'Bilateral' in f and f.endswith('.xlsx')]
if excel_files:
    print(f"  📋 Generated Excel files:")
    for file in excel_files:
        size = os.path.getsize(file) / 1024  # Size in KB
        print(f"     ✓ {file} ({size:.1f} KB)")
else:
    print("  💡 Excel files will be generated when data extraction completes")

print(f"\n📈 How to Use This Script:")
print("  1. Open cell 2 and modify the country codes:")
print("     - debtorCountry = 'IND'  # Change to debtor country code")
print("     - creditorCountry = 'CHN'  # Change to creditor country code")
print("  2. Run all cells sequentially")
print("  3. Data will be extracted and saved to Excel")
print("  4. A visualization chart will be generated")

print(f"\n💾 Data Format:")
print("  - Columns: year, creditor, debtor, indicator, data")
print("  - Unit: Current US$ (millions)")
print("  - Time period: Various years from World Bank database")

print(f"\n{'=' * 70}")
print("✓ Project setup and execution completed!")
print(f"{'=' * 70}\n")


INTERNATIONAL DEBT STATISTICS PROJECT - EXECUTION REPORT

📊 Project Objective:
  Analyze PPG (Public and Publicly Guaranteed) bilateral debt data
  obtained from the World Bank API

🔍 Data Source:
  World Bank International Debt Statistics (IDS) - Source 6
  Indicator: DT.DOD.BLAT.CD (PPG, bilateral debt)
  Updated: 2025-12-03

✓ Project Execution Summary:
  1. ✓ Loaded required Python packages (pandas, requests, matplotlib, seaborn)
  2. ✓ Configured notebook with Python 3.0.5
  3. ✓ Connected to World Bank API
  4. ✓ Fetched bilateral debt data for India
  5. ✓ Found bilateral lending data between countries

📁 Output Files:


NameError: name 'os' is not defined